# 多言語翻訳モデル NLLB で翻訳する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/27_machine_translation/translation_nllb.ipynb)

NLP クックブック 27-03 ／ 機械翻訳

日本語と英語、あるいは 200 言語の任意の組み合わせで機械翻訳をしたい。API を使わず手元で動かしたい。

In [1]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

name = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name).eval()

def translate(texts, src, tgt, num_beams=4):
    tokenizer.src_lang = src
    batch = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=256)
    with torch.no_grad():
        out = model.generate(**batch, forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt),
                             num_beams=num_beams, max_new_tokens=256)
    return tokenizer.batch_decode(out, skip_special_tokens=True)

ja = ["吾輩は猫である。名前はまだ無い。", "機械翻訳の品質は、ここ数年で大きく向上した。"]
for s, t in zip(ja, translate(ja, "jpn_Jpan", "eng_Latn")):
    print(s, "→", t)

吾輩は猫である。名前はまだ無い。 → I'm a cat. I don't have a name yet.
機械翻訳の品質は、ここ数年で大きく向上した。 → The quality of machine translation has improved dramatically in recent years.


In [2]:
en = ["Natural language processing is a subfield of artificial intelligence.", "Could you tell me how to get to the station?"]
for tgt in ["jpn_Jpan", "zho_Hans", "kor_Hang", "fra_Latn"]:
    print(tgt, translate(en, "eng_Latn", tgt))

jpn_Jpan ['自然言語処理は人工知能のサブフィールドです', '駅まで行く方法を教えてくれないか?']
zho_Hans ['自然语言处理是人工智能的一个子领域.', '你能告诉我如何到车站吗?']
kor_Hang ['자연어 처리는 인공지능의 하위 분야입니다.', '경찰서에 어떻게 갈 수 있는지 말해 주시겠어요?']
fra_Latn ["Le traitement du langage naturel est un sous-field de l'intelligence artificielle.", 'Pouvez-vous me dire comment aller au poste ?']


- 言語は FLORES-200 のコード（`jpn_Jpan`、`eng_Latn`、`zho_Hans` など）で指定する。出力言語は、デコーダの最初のトークンを強制することで決まる。
- 文単位で学習されたモデルなので、長い文章は文に分割してから 1 文ずつ訳す。
- 精度を上げるなら `facebook/nllb-200-1.3B` や `nllb-200-3.3B`（GPU 推奨）。NLLB のライセンスは CC-BY-NC（非商用）である点に注意。
- 日英・英日なら、手元で動く小さな LLM に翻訳させる方法もある。NLLB との比較や、出力の後処理、前後の文脈の与え方は、この章の「LLM で日英・英日翻訳して NLLB と比べる（ゼロショット、few-shot、前後の文脈）」を参照。